# 17 · An LSP-guided coding agent (MicroVM)

Instead of "run it and read the traceback", the agent inspects the code with a
**language server** — symbols and completions — to ground its edit *before*
executing, then confirms with the test suite.

**Key concepts**
- **`sbx.lsp.create_server(language, project_root)`** — starts a real language
  server (`pylsp`) in the VM; `did_open`, `document_symbols`, `completions`
  speak LSP over JSON-RPC and return structured code intelligence.
- **Grounding the model** — feeding `document_symbols` into the prompt gives the
  model the project's real structure (names, line numbers) rather than guesses.
- **`sbx.git` / `commands`** — clone/edit/run the project in the same VM.
- Needs the **`microvm`** backend (a language server is an OS process).

**Prerequisites**
- AWS creds + region; image ARN in `RAITBOX_MICROVM_IMAGE` (with `pylsp` installed
  — e.g. built via `Image().pip_install("python-lsp-server")`, see notebook 21).
- Bedrock model access for `RAITBOX_BEDROCK_MODEL`.

In [ ]:
%pip install -q raitbox

### Configuration

In [ ]:
import os

import boto3

from raitbox import Sandbox

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

PROJECT = "/work"

## 1 · Seed a tiny project with a bug
`add` is correct; `average` divides by the wrong length. A test encodes the
intended behavior.

In [ ]:
CALC = """def add(a, b):
    return a + b


def average(values):
    # BUG: off-by-one in the denominator
    return sum(values) / (len(values) - 1)
"""

TEST = """from calc import average

def test_average():
    assert average([2, 4, 6]) == 4
"""

sbx = Sandbox.create(backend="microvm", template=IMAGE_ARN)
sbx.files.write(f"{PROJECT}/calc.py", CALC)
sbx.files.write(f"{PROJECT}/test_calc.py", TEST)
print("failing before fix:")
print(sbx.commands.run(f"cd {PROJECT} && python -m pytest -q || true").stdout)

## 2 · Ask the language server what's in the file
`document_symbols` is real LSP output — the functions the model may edit, with
their kinds and locations.

In [ ]:
lsp = sbx.lsp.create_server("python", PROJECT)
lsp.did_open(f"{PROJECT}/calc.py")
symbols = lsp.document_symbols(f"{PROJECT}/calc.py")
for s in symbols:
    print(f"kind={s.kind:>2}  {s.name}")

## 3 · Ground the model with the symbols, get a fix

In [ ]:
symbol_names = ", ".join(s.name for s in symbols) or "(none)"
prompt = f"""File calc.py defines these top-level symbols: {symbol_names}.
Current source:
{sbx.files.read(f"{PROJECT}/calc.py")}

The test `test_average` expects average([2,4,6]) == 4 but it fails.
Return the corrected full contents of calc.py only (no fences)."""

resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 1024, "temperature": 0},
)
fixed = resp["output"]["message"]["content"][0]["text"].strip()
if fixed.startswith("```"):
    fixed = fixed.split("```")[1].removeprefix("python").strip()
sbx.files.write(f"{PROJECT}/calc.py", fixed)
print(fixed)

## 4 · Re-open in the LSP (sanity) and run the suite

In [ ]:
lsp.did_open(f"{PROJECT}/calc.py")  # re-sync the edited document
result = sbx.commands.run(f"cd {PROJECT} && python -m pytest -q")
print(result.stdout)
print("PASSED" if result.exit_code == 0 else "still failing")

lsp.stop()
sbx.kill()

## Recap
- `document_symbols` gave the model the project's real structure to edit against.
- The same server also offers `completions(path, Position(line, char))` and
  `workspace_symbols(query)` for richer navigation.

**Related:** `06-iterative-self-repair` (run-only loop), `02-agent-feedback-loop`.